# RobusPredictor — Demostración (V130)
**Versión librería:** 1.3.0   **Autores:** Paula Frías · Sebastián Valdivia

**Repositorio:** https://github.com/paufriasest/Robus-Predictor

---
## Sección 0: Preparación del entorno
### 0.1 — Verificación de dependencias

Comprueba que todas las librerías necesarias estén disponibles en el entorno activo.
Si alguna falta, muestra el comando exacto para instalarla en la terminal.

In [ ]:
#Verificación de dependencias
import importlib

DEPENDENCIAS = ["numpy", "pandas", "sklearn", "openpyxl"]

faltantes = []
print("Verificando dependencias...")
for modulo in DEPENDENCIAS:
    pip_nombre = "scikit-learn" if modulo == "sklearn" else modulo
    if importlib.util.find_spec(modulo) is not None:
        print("  OK  " + pip_nombre)
    else:
        print("  FALTA  " + pip_nombre)
        faltantes.append(pip_nombre)

if faltantes:
    print("\nEjecuta esto en tu terminal con el entorno activo y reinicia el kernel:")
    print("  pip install " + " ".join(faltantes))
    raise ImportError("Faltan dependencias: " + str(faltantes))

print("\nTodas las dependencias disponibles.")

### 0.2 — Importación de la librería (carpeta local)

La librería se distribuye como la carpeta **`/Producto/robuspredictor/` ubicada junto a este notebook**. 
No requiere instalación ni clonar el repositorio: basta con que la carpeta esté en el mismo directorio que este archivo `.ipynb`.

In [ ]:
# Importación de la librería (modelo carpeta-copiable)
# La librería se usa copiando la carpeta `/Producto/robuspredictor/` junto a este notebook

import os, sys

RUTA_BASE = os.getcwd()
RUTA_PRODUCTO = os.path.join(RUTA_BASE, "Producto")

rutas_a_incluir = [RUTA_BASE, RUTA_PRODUCTO]

for ruta in rutas_a_incluir:
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

# 3. Intentar la importación
try:
    from robuspredictor import RobusPredictor, __version__
    print(f"OK  RobusPredictor v{__version__} importado correctamente.")
except ImportError as e:
    raise ImportError(
        "No se encontró el módulo 'robuspredictor'.\n"
        "Asegúrate de que la carpeta 'robuspredictor' esté ubicada en una de las siguientes rutas:\n"
        f" 1. {RUTA_BASE}\n"
        f" 2. {RUTA_PRODUCTO}\n"
        f"Detalle técnico: {e}"
    )

### 0.3 — Importaciones generales

Librerías estándar utilizadas a lo largo del notebook.

In [ ]:
# Importaciones generales
import pandas as pd
import numpy as np
import time
import warnings
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
)

print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print("sklearn      : importado")
print("Entorno listo.")

---
## Sección 1: Carga de datos

### 1.1 — Rutas de los archivos CSV

> **Modifica aquí** los nombres y rutas a tus archivos de entrenamiento y validación.

In [ ]:
#  Carga de datos
TRAINING_PATH   = "DATOS_ENTRENAMIENTO.csv" # Debe estar en el mismo nivel del notebook en este ejemplo de ruta.
VALIDATION_PATH = "DATOS_VALIDACION.csv"    # Debe estar en el mismo nivel del notebook en este ejemplo de ruta.
# ─────────────────────────────────────────────────────────────

ENTRENAMIENTO = pd.read_csv(TRAINING_PATH)
VALIDACION    = pd.read_csv(VALIDATION_PATH)

print(f"Entrenamiento : {ENTRENAMIENTO.shape[0]:,} filas × {ENTRENAMIENTO.shape[1]} columnas")
print(f"Validación    : {VALIDACION.shape[0]:,} filas × {VALIDACION.shape[1]} columnas")
print(f"\nColumnas disponibles: {list(ENTRENAMIENTO.columns)}")

### 1.2 — Definir variables predictoras y target

> **Modifica aquí** los nombres de las variables (columnas) según el dataset.

In [ ]:
# Definir features y target

features = ['var1', 'var2', 'var3', 
]
target = "INTENSIDAD_4H"
# ─────────────────────────────────────────────────────────────

X_train = ENTRENAMIENTO[features]
y_train = ENTRENAMIENTO[target]

X_valid = VALIDACION[features]
y_valid = VALIDACION[target]

y_valid_arriendo = VALIDACION["ARRIENDO"]

print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"X_valid : {X_valid.shape}")
print(f"y_valid : {y_valid.shape}")

print("\nPrimeras filas del conjunto de entrenamiento:")
display(X_train.head(10))

---
## Sección 2: Configuración del modelo

### Descripción de parámetros

| Parámetro | Descripción |
|-----------|-------------|
| `n_min` | Tamaño mínimo de registros por cubo. Valores bajos aumentan la granularidad pero reducen la estabilidad. |
| `n_max` | Tamaño máximo de registros por cubo. Actúa como control de calidad del particionamiento. |
| `n_dom` | Número de dominios internos (mínimo 2). El dominio 1 construye el árbol; los demás validan estabilidad. A mayor valor, mayor exigencia. |
| `mean_min` | Promedio mínimo aceptable del target en un cubo estable. Define el piso del rango de interés. |
| `mean_max` | Promedio máximo aceptable del target en un cubo estable. Define el techo del rango de interés. |
| `std_min` | Desviación estándar mínima del target. En la mayoría de los casos se mantiene en 0. |
| `std_max` | Desviación estándar máxima del target. Controla la homogeneidad interna del cubo. |
| `use_default_value` | Si `True` (default), los registros en cubos inestables reciben `default_value`. Si `False`, reciben el promedio del propio cubo, produciendo un espacio vectorial sin ceros. |
| `default_value` | Valor asignado a registros en cubos inestables cuando `use_default_value=True`. Por convención se usa 0. |


> **⚙️ Modifica los valores según tu dataset antes de entrenar.**

In [ ]:
#Parámetros del modelo (modifica según tu dataset)

PARAMS = dict(
    n_min             = 10,
    n_max             = 20,
    n_dom             = 2,
    mean_min          = 0.1,
    mean_max          = 10.0,
    std_min           = 0.0,
    std_max           = 10.0,
    use_default_value = True,   # False → cubos inestables usan promedio del cubo
    default_value     =  0,     # valor para registros sin cubo estable (si use_default_value=True)
    verbose           = False   
)
# ─────────────────────────────────────────────────────────────

print("Parámetros configurados:")
for k, v in PARAMS.items():
    print(f"  {k:<20} = {v}")

In [ ]:
#Entrenamiento del modelo
modelo = RobusPredictor(**PARAMS)

print("Entrenando modelo...")
t0 = time.perf_counter()
modelo.fit(X_train, y_train)
t1 = time.perf_counter()

execution_time = t1 - t0

print(f"\nEntrenamiento completado en {execution_time:.2f} segundos.")
print(f"  Dominios generados    : {len(modelo.domains)}")
print(f"  Cubos estables        : {len(modelo.stable_cubes)}")
print(f"  Zonas rojas           : {len(modelo.red_zones)}")

---
## Sección 3: Predicción y cobertura

Se generan las predicciones sobre el conjunto de validación.
La cobertura indica qué proporción de registros cayó en un cubo estable.

In [ ]:
# Predicción
y_pred  = modelo.predict(X_valid)
n_total = len(y_pred)

if modelo.use_default_value:
    n_cubiertos = (y_pred != modelo.default_value).sum()
    pct         = n_cubiertos / n_total * 100
    print("Cobertura del modelo (validación):")
    print(f"  Total registros          : {n_total:,}")
    print(f"  Con cubo estable         : {n_cubiertos:,}  ({pct:.1f} %)")
    print(f"  Con valor por defecto    : {n_total - n_cubiertos:,}  ({100 - pct:.1f} %)")
else:
    n_cubiertos = n_total
    pct         = 100.0
    print("Cobertura del modelo (validación):")
    print(f"  Total registros          : {n_total:,}")
    print(f"  use_default_value=False  → todos los registros reciben un valor predicho.")
    print(f"  Cubos estables en modelo : {len(modelo.stable_cubes)}")
    print(f"  Zonas rojas en modelo    : {len(modelo.red_zones)}")

---
## Sección 4: Diagnóstico de cubos

> Las celdas de esta sección son de diagnóstico visual.
> **Pueden saltarse sin afectar el flujo principal.**
> Con datasets grandes se recomienda omitirlas
> para evitar alto consumo de recursos.


---
### 4.1  — Exportar Checkpoint

Genera el archivo Excel de auditoría con una fila por cubo. Incluye promedios de las
variables predictoras por dominio, métricas del target por dominio, estadísticas sobre
el conjunto de validación y el árbol de cortes completo.

Pasar `X_valid` e `y_valid` agrega al Excel las columnas:
- `n_validacion` — registros del dataset de validación en cada cubo
- `prom_target_validacion` — promedio del target real en validación por cubo
- `std_target_validacion` — desviación estándar del target real en validación
- `prom_target_consolidado` — promedio global (dom1 + dom2 + validación)


In [ ]:
# Exportar checkpoint Excel

modelo.export_checkpoint(
    X_valid=X_valid,
    y_valid=y_valid,
    file_name="checkpoint_robuspredictor",
    file_format="xlsx"
)

ck= modelo.checkpoint
cubes_df = ck["cubes_checkpoint"]

print(f"Checkpoint")
print(f"  Hojas: cubes_checkpoint | cuts_checkpoint | summary")
print(f"  Total cubos : {len(cubes_df)}")
print(f"  Estables    : {cubes_df['estable'].sum()}")
print(f"  Zonas rojas : {(cubes_df['estable'] == 0).sum()}")
print()
cols_vista = [
    c for c in cubes_df.columns
    if not c.startswith("indices_")
    and not c.startswith("prom_dominios_entrenamiento_detalle")
]
display(cubes_df[cols_vista].head(10))

### 4.2 — Exportar scoring de predicción (trazabilidad por registro)

A diferencia del checkpoint general (Sección 5.1), que resume la información por cubo,
este export muestra qué ocurrió con **cada fila individual** entregada al modelo.
Permite auditar registro a registro la predicción asignada y su resultado real.

#### Columnas del Excel de scoring:

| Columna | Descripción |
|---------|-------------|
| `id_registro` | Índice del registro (1-based); el total equivale al número de filas de validación |
| `var1, var2, ...` | Variables originales utilizadas en el modelo |
| `target` | Valor real de `INTENSIDAD_4H` (`y_valid`). Solo para análisis |
| `id_cubo` | Identificador del cubo asignado al registro |
| `estable` | 1 si el cubo fue estable, 0 si es zona roja |
| `promedio_cubo` | Promedio del target en el cubo correspondiente |
| `prediccion_aplicada` | Valor final asignado al registro. Si el cubo es estable recibe el promedio; si no, `default_value` |
| `motivo_rechazo` | Razón por la que el cubo fue zona roja (vacío si es estable) |
| `arriendo_segun_predict` | Clasificación generada a partir de `prediccion_aplicada`: 1 si supera el umbral 1.5, 0 si no |
| `ARRIENDO_REAL` | Valor real observado de arriendo en validación (1 = hubo arriendo, 0 = no) |
| `acierto_del_modelo_de_acuerdo_arriendo_predict` | 1 si `arriendo_segun_predict == 1` y `ARRIENDO_REAL == 1`; 0 en cualquier otro caso |


In [ ]:
# Exportar scoring de predicción
scoring_df = modelo.export_prediction_checkpoint(
    X_valid=X_valid,
    y_valid=y_valid,
    dato_real=y_valid_arriendo,  # variable binaria para comparacion (en este caso es ARRIENDO)
    file_name="scoring_robuspredictor",
    file_format="xlsx",
)

print(f"Scoring")
print(f"  Registros : {len(scoring_df):,}")
print()

cols_vista = [c for c in scoring_df.columns if c != "motivo_rechazo"]
display(scoring_df[cols_vista].head(10))

### 4.3 — DataFrame: Resumen de cubos utilizados en predicción

A diferencia de los demás exports, este genera un DataFrame que permite
identificar, por cada `id_cubo`, el rango (valor mínimo y máximo) que
tomó cada variable predictora entre los registros que componen ese cubo,
junto con el valor de predicción asignado.

Se obtiene llamando a `modelo.export_dataframe_cubes()`.

#### Columnas del DataFrame

| Columna | Descripción |
|---------|-------------|
| `ID` | Identificador del cubo (`id_cubo`) |
| `<variable>_min` | Valor mínimo de esa variable entre los registros del cubo (una columna por cada variable predictora) |
| `<variable>_max` | Valor máximo de esa variable entre los registros del cubo (una columna por cada variable predictora) |
| `Pred` | Valor de predicción asignado al cubo |

In [ ]:
cubes_df = modelo.export_dataframe_cubes()
display(cubes_df.head())

### 4.4 — DataFrame: Grilla completa de cubos (reglas por cubo)

A diferencia de la sección 5.3 —que resume los rangos *observados* de los cubos usados en la
última predicción—, este export devuelve la **grilla completa aprendida durante el
entrenamiento**: una fila por cada cubo final del modelo (estables y zonas rojas), con la
**regla exacta** que define ese cubo a partir de los cortes por mediana.

Sirve para auditar la grilla o reproducirla a mano (en Excel, pandas o cualquier lenguaje),
sin depender de la librería.

Se obtiene llamando a `modelo.export_cubes_grid()`.

#### Columnas del DataFrame

| Columna | Descripción |
|---------|-------------|
| `cube_id` | Identificador público del cubo (ej. `CUBE_001`) |
| `group_id` | Identificador interno según la ruta de cortes (ej. `LLRL`) |
| `estable` | 1 si el cubo es estable (zona verde), 0 si es zona roja |
| `Pred` | Predicción asignada al cubo (promedio de promedios si es estable; `default_value` si es zona roja y `use_default_value=True`) |
| `regla_completa` | Condición que define el cubo (ej. `var1 <= 0.5 AND var2 > 1.2 ...`), reproducible manualmente |

In [ ]:
# export_cubes_grid() -> DataFrame con UNA fila por cubo final (estables + zonas rojas).
# Cada fila incluye la regla completa del cubo, por lo que permite auditar o reproducir
# la grilla aprendida sin necesidad de la librería.
cube_grid = modelo.export_cubes_grid()
display(cube_grid.head())

### 4.5 — DataFrame: cubo y predicción por registro

Esta sección arma un DataFrame (aquí llamado `resultado`) que, para **cada registro** del
conjunto de validación, agrega dos columnas:

- `cube_id` — el cubo al que pertenece el registro, mediante `modelo.predict_cubes(X_valid)`.
- `pred` — el valor predicho para ese registro, mediante `modelo.predict(X_valid)`.

Es la forma, al estilo scikit-learn, de pegar la asignación de cubo y la predicción
directamente como columnas nuevas de un DataFrame existente.

In [ ]:
# Copiamos X_valid para no alterar el DataFrame original.
resultado = X_valid.copy()

# predict_cubes(): id público del cubo al que cae cada registro.
resultado["cube_id"] = modelo.predict_cubes(X_valid)

# predict(): valor de predicción asignado a cada registro.
resultado["pred"] = modelo.predict(X_valid)

resultado.head()

### 4.6 — Guardar el modelo entrenado (joblib)

Guarda el modelo **ya entrenado** en un archivo `.pkl` para **reutilizarlo más tarde sin volver a llamar `fit()`**.

`joblib.dump()` serializa el objeto completo a disco; `joblib.load()` lo reconstruye en memoria y permite llamar `predict()` directamente.

Se guarda el **modelo completo** : el `.pkl` incluye los dominios de entrenamiento, por lo que el modelo recargado puede no solo predecir, sino también regenerar `export_checkpoint` y `export_dataframe_cubes`.

> **Importante:** el `.pkl` solo se recarga en un entorno con la **misma versión de la librería** (`robuspredictor`) y de Python/pandas compatibles. Guárdalo junto a la carpeta en la misma carpeta donde se ejecute este notebook.

In [ ]:
#  Guardar el modelo entrenado (joblib) para reutilizarlo sin reentrenar
import joblib, os

RUTA_MODELO = "modelo_robuspredictor.pkl"

joblib.dump(modelo, RUTA_MODELO)
size_mb = os.path.getsize(RUTA_MODELO) / 1024**2
print(f"Modelo guardado en: {RUTA_MODELO}  ({size_mb:.1f} MB)")

### 4.7 — Cargar el modelo guardado (joblib)

In [ ]:
#  Cargar modelo entrenado (joblib) para reutilizarlo sin reentrenar
import joblib, os
modelo = joblib.load(RUTA_MODELO) # Se asume que el archivo "modelo_robuspredictor.pkl" está en el mismo nivel del notebook.

---
## Sección 5: Métricas de evaluación

### 5.1 — Métricas de negocio

Evalúa el desempeño del modelo sobre el conjunto de validación desde dos perspectivas:

- **MAE sobre cubos estables** — error absoluto medio entre el valor predicho y el valor real de `INTENSIDAD_4H`, calculado únicamente sobre los registros asignados a un cubo estable.
- **Precisión Top N%** — de los registros con predicción más alta (el N% superior), qué proporción corresponde efectivamente a `ARRIENDO = 1`. Es la métrica principal de negocio. Configurable mediante `TOP_PCT`.-

> ** Modifica `TOP_PCT`** para cambiar el porcentaje de corte (0.05 = Top 5%).

In [ ]:
#  Métricas de negocio (MAE, Precisión Top N% y cubos estables)
from robuspredictor import calcular_mae

TOP_PCT = 0.05  # 0.05 = Top 5% | 0.01 = Top 1% | 0.10 = Top 10%

y_pred  = modelo.predict(X_valid)
n_total = len(y_pred)

mae_total = calcular_mae(y_valid, y_pred)

if modelo.use_default_value:
    mask_cubiertos = y_pred != modelo.default_value
else:
    mask_cubiertos = pd.Series([True] * len(y_pred), index=y_pred.index)

mae_cubiertos = calcular_mae(y_valid[mask_cubiertos], y_pred[mask_cubiertos])

print("── MAE (INTENSIDAD_4H) ──────────────────────────────────")
print(f"  MAE total (incl. zonas no estables) : {mae_total:.4f}")
print(f"  MAE solo cubos estables             : {mae_cubiertos:.4f}")
print(f"  Registros cubiertos                 : {mask_cubiertos.sum():,} / {len(y_pred):,}")

precision = modelo.best_percentage(y_valid_arriendo, top_pct=TOP_PCT)

tasa_base = y_valid_arriendo.mean()
n_top     = max(1, int(len(y_pred) * TOP_PCT))

print(f"\n── Precisión Top {int(TOP_PCT*100)}% (ARRIENDO) ──────────────────────────")
print(f"  Registros en Top {int(TOP_PCT*100)}%                 : {n_top:,}")
print(f"  Precisión Top {int(TOP_PCT*100)}%                    : {precision:.4f}  ({precision*100:.1f}%)")
print(f"  Tasa base ARRIENDO=1                : {tasa_base:.4f}  ({tasa_base*100:.1f}%)")
print(f"  Cubos estables                      : {len(modelo.stable_cubes)}")
print()
print("  Interpretación:")
print(f"    De los {n_top:,} registros con predicción más alta,")
print(f"    el {precision*100:.1f}% corresponde a ARRIENDO=1.")

### 5.2 — Métricas estadísticas complementarias

Métricas de regresión estándar para comparar RobusPredictor contra otros modelos.

| Métrica | Descripción |
|---------|-------------|
| **RMSE** | Raíz del error cuadrático medio. Penaliza errores grandes con mayor peso que el MAE. |
| **MedAE** | Error absoluto mediano. Robusto frente a valores extremos. |
> R² y MAPE no se incluyen: R² se distorsiona cuando muchos registros reciben `default_value = 0`; MAPE produce valores sin sentido con target cercano a cero.

In [ ]:
# Métricas estadísticas complementarias (sklearn)
rmse_sk  = np.sqrt(mean_squared_error(y_valid, y_pred))
medae_sk = median_absolute_error(y_valid, y_pred)

print("-" * 52)
print("  MÉTRICAS ESTADÍSTICAS — ROBUS PREDICTOR")
print("-" * 52)
print(f"  Tiempo fit()     : {execution_time:.2f} s")
print("-" * 52)
print(f"  RMSE             : {rmse_sk:.4f}")
print(f"  MedAE            : {medae_sk:.4f}")
print("-" * 52)

---
## Notas finales

**Flujo mínimo para ejecutar el modelo y obtener los entregables:**

```
Sección 1 (Carga) → Sección 2 (Configuración) → Entrenamiento (fit) → Sección 3 (Predicción) → Sección 4.1 (Checkpoint) → Sección 4.2 (Scoring)
```

**Para ajustar el modelo:** modifica los valores de la **Sección 2** y vuelve a ejecutar el
entrenamiento (`fit`) y la **Sección 3** (predicción); luego regenera los exports de la
**Sección 4** que necesites.

```
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣤⠶⠶⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⣀⣀⣀⣀⣿⠁⠀⠀⢹⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⢀⣀⠀⠀⠀⠀⣀⣤⠴⠖⠛⠛⠋⠉⠉⠉⠙⠋⠀⠀⠀⠘⠁⣀⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⡴⠛⠉⠛⣦⣴⠟⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠛⠳⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠸⡇⠀⠀⠀⠛⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⣹⡦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⣸⠟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⡧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀
⠀⠀⢀⡾⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠰⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀
⠀⢠⡿⠁⠀⠀⠀⠀⠀⠀⠀⠀⠐⠛⠳⠆⠀⠀⠀⠀⠀⠀⠀⢠⣾⠛⢳⣆⠀⠀⠄⠐⡀⣄⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀
⠀⡿⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣤⣄⡀⠀⠀⠀⠀⠀⠀⠸⣿⣷⣻⡟⠈⡆⣸⢸⡇⠃⣁⡀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀
⢸⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⣿⣁⣼⣿⠀⠀⠀⠀⠀⠀⠀⠀⠉⠁⠘⡀⠁⢈⣤⡴⠋⠉⠉⠙⢦⣀⣿⠀⠀⠀⠀⠀⠀⠀⠀
⢸⡇⠀⠀⠀⠀⠀⠀⠀⠀⠠⠄⠀⠻⠷⠾⠋⠀⠀⡀⠀⣶⣀⡶⠀⠀⠀⠀⢀⣰⠏⠀⣠⡴⠶⠶⣦⡈⢿⣿⠳⣆⠀⠀⠀⠀⠀⠀
⠘⣇⠀⠀⠀⠀⠀⠀⡐⡇⣴⢠⡆⡖⠀⠀⠀⣀⡀⠟⠟⢋⢉⡀⠀⠀⠀⢀⡿⠁⢀⡾⠋⠀⠙⢳⣄⠙⢷⡈⠳⠿⢤⣤⡀⠀⠀⠀
⠀⢿⡄⠀⠀⠀⠀⠀⠻⠇⠙⠈⠁⠠⠊⠀⠘⠤⠗⠀⠀⠈⠉⣠⠤⠶⠛⠉⠀⣠⡞⠀⠀⠀⠀⠀⠙⣆⠈⢷⡀⠠⠀⠀⠙⣦⡀⠀
⠀⠘⣷⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡁⠀⣴⠟⠋⠉⠉⠀⠀⠀⠀⠀⠀⠀⢸⡆⠘⠷⢦⣤⡈⠂⠹⣧⡀
⠀⠀⠈⢿⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣧⠀⢿⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠓⠒⠳⣦⢹⡆⠀⠈⣷
⠀⠀⠀⠀⠙⢷⣤⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⣤⠶⠷⣦⠈⢷⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⢏⣼⠃⠀⢠⡟
⠀⠀⠀⠀⠀⠀⠀⠉⢫⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣠⡟⠀⢨⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡶⠋⣡⡟⠁⠀⢀⣼⠁
⠀⠀⠀⠀⠀⠀⠀⠀⠈⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢸⡇⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠀⣿⠀⠀⣴⠋⠁⠀
⠀⠀⠀⠀⠀⠀⠀⠀⣠⣿⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⠀⠸⣆⠀⠀⠀⢀⣠⡤⢦⣀⠀⠀⢠⡿⠀⡟⠀⢠⡏⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠘⣧⠀⢹⣧⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⢷⣄⠀⠉⠛⠛⢛⠏⢁⣀⠀⠙⠳⢦⣤⣤⠞⠁⠀⣸⠇⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠉⠀⠙⠳⠦⣤⡄⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠒⢒⡖⠚⠚⠋⠉⠛⢦⣤⣀⣀⣀⣀⣤⠾⠋⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢿⡀⠀⢸⡗⠒⠶⠶⠒⢶⡆⢀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠉⠉⠀⠀⠀⠀⠀⠀⠀
⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⠒⠛⠁⠀⠀⠀⠀⠈⠛⠛⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

```